<a href="https://colab.research.google.com/github/talhanoor23/my-projects/blob/main/AI_Powered_Deepfake_Video_Detection_System/Deepfake_Video_Detection(Improved_Version).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# import kagglehub

# # Download latest version
# path = kagglehub.dataset_download("yuvrajpaikhot/extracted-deepfake-frames")

# print("Path to dataset files:", path)

In [ ]:
# Create a .kaggle folder
!mkdir -p ~/.kaggle
# Move the kaggle.json file there
!mv kaggle.json ~/.kaggle/
# Set the permissions
!chmod 600 ~/.kaggle/kaggle.json

In [ ]:
!kaggle datasets download -d yuvrajpaikhot/extracted-deepfake-frames

Dataset URL: https://www.kaggle.com/datasets/yuvrajpaikhot/extracted-deepfake-frames
License(s): CC0-1.0


In [ ]:
!unzip /content/extracted-deepfake-frames.zip

In [ ]:
# import os
# import shutil

# # Original dataset path
# source_dir = '/content/train'

# # New destination path
# dest_dir = '/content/train_restructured'

# # Create new structure
# fake_dir = os.path.join(dest_dir, 'fake')
# real_dir = os.path.join(dest_dir, 'real')
# os.makedirs(fake_dir, exist_ok=True)
# os.makedirs(real_dir, exist_ok=True)

# # Iterate through original folders
# for folder_name in os.listdir(source_dir):
#     folder_path = os.path.join(source_dir, folder_name)

#     # Skip non-directory items or already structured folders
#     if not os.path.isdir(folder_path) or folder_name in ['fake', 'real']:
#         continue

#     # Copy based on naming
#     if folder_name.startswith('fake_train_'):
#         shutil.copytree(folder_path, os.path.join(fake_dir, folder_name))
#     elif folder_name.startswith('real_train_'):
#         shutil.copytree(folder_path, os.path.join(real_dir, folder_name))

# print("Copied and restructured dataset to /content/train_restructured/")


In [ ]:
#ye uper walay ki tarha hi hae abs isme 250 videos copy ki hain from each real and fake.

import os
import shutil

# Original dataset path
source_dir = '/content/train'

# New destination path
dest_dir = '/content/train_restructured'

# Create new structure
fake_dir = os.path.join(dest_dir, 'fake')
real_dir = os.path.join(dest_dir, 'real')
os.makedirs(fake_dir, exist_ok=True)
os.makedirs(real_dir, exist_ok=True)

# List of all video folders in the source directory
all_folders = [folder_name for folder_name in os.listdir(source_dir) if os.path.isdir(os.path.join(source_dir, folder_name))]

# Select only 250 fake and 250 real video folders
fake_folders = [folder for folder in all_folders if folder.startswith('fake_train_')][:400]
real_folders = [folder for folder in all_folders if folder.startswith('real_train_')][:254]

# Copy selected fake videos to the fake directory
for folder_name in fake_folders:
    folder_path = os.path.join(source_dir, folder_name)
    shutil.copytree(folder_path, os.path.join(fake_dir, folder_name))

# Copy selected real videos to the real directory
for folder_name in real_folders:
    folder_path = os.path.join(source_dir, folder_name)
    shutil.copytree(folder_path, os.path.join(real_dir, folder_name))

print(f"Copied and restructured dataset to {dest_dir} (250 fake and 250 real videos).")


Copied and restructured dataset to /content/train_restructured (250 fake and 250 real videos).


In [ ]:
import os

# Path to old dataset
old_dataset_path = '/content/train'

# Count fake folders
fake_folders = [d for d in os.listdir(old_dataset_path) if d.startswith('fake_train_')]
real_folders = [d for d in os.listdir(old_dataset_path) if d.startswith('real_train_')]

print(f"Old Dataset - Fake folders: {len(fake_folders)}")
print(f"Old Dataset - Real folders: {len(real_folders)}")


Old Dataset - Fake folders: 2147
Old Dataset - Real folders: 254


In [ ]:
# Path to new dataset
new_fake_path = '/content/train_restructured/fake'
new_real_path = '/content/train_restructured/real'

# Count folders in each
num_fake = len([d for d in os.listdir(new_fake_path) if os.path.isdir(os.path.join(new_fake_path, d))])
num_real = len([d for d in os.listdir(new_real_path) if os.path.isdir(os.path.join(new_real_path, d))])

print(f"New Dataset - Fake folders: {num_fake}")
print(f"New Dataset - Real folders: {num_real}")


New Dataset - Fake folders: 400
New Dataset - Real folders: 254


In [ ]:
import shutil

# Zip the folder (creates train_restructured.zip)
shutil.make_archive('/content/train_restructured', 'zip', '/content/train_restructured')

'/content/train_restructured.zip'

In [ ]:
import os
import random
import numpy as np
import tensorflow as tf
from tensorflow.keras.utils import Sequence
from tensorflow.keras.preprocessing.image import load_img, img_to_array
import albumentations as A  # Powerful augmentation library
from albumentations.core.composition import OneOf

class FrameSequenceGenerator(Sequence):
    def __init__(self, dataset_path, batch_size=4, num_frames=10, target_size=(224, 224), shuffle=True, augment=False):
        self.dataset_path = dataset_path
        self.batch_size = batch_size
        self.num_frames = num_frames
        self.target_size = target_size
        self.shuffle = shuffle
        self.augment = augment

        self.video_paths = []
        self.labels = []

        for label, folder in enumerate(['real', 'fake']):
            folder_path = os.path.join(dataset_path, folder)
            for video_folder in os.listdir(folder_path):
                full_path = os.path.join(folder_path, video_folder)
                if os.path.isdir(full_path):
                    self.video_paths.append(full_path)
                    self.labels.append(label)

        self.on_epoch_end()

        # Define augmentations (only if needed)
        if self.augment:
            self.augmentations = A.Compose([
                OneOf([
                    A.RandomBrightnessContrast(p=0.5),
                    A.HueSaturationValue(p=0.5),
                    A.RGBShift(p=0.5),
                ], p=0.8),
                A.HorizontalFlip(p=0.5),
                A.RandomCrop(height=200, width=200, p=0.5),
                A.Resize(height=self.target_size[0], width=self.target_size[1])
            ])
        else:
            self.augmentations = None

    def __len__(self):
        return int(np.ceil(len(self.video_paths) / self.batch_size))

    def __getitem__(self, idx):
        batch_video_paths = self.video_paths[idx * self.batch_size:(idx + 1) * self.batch_size]
        batch_labels = self.labels[idx * self.batch_size:(idx + 1) * self.batch_size]

        batch_x = []
        batch_y = []

        for video_path, label in zip(batch_video_paths, batch_labels):
            frames = self.load_frames(video_path)
            batch_x.append(frames)
            batch_y.append(label)

        return np.array(batch_x), np.array(batch_y)

    def on_epoch_end(self):
        if self.shuffle:
            temp = list(zip(self.video_paths, self.labels))
            random.shuffle(temp)
            self.video_paths, self.labels = zip(*temp)

    def load_frames(self, video_folder):
        frame_files = sorted(os.listdir(video_folder))
        total_frames = len(frame_files)

        if total_frames >= self.num_frames:
            # Randomly select num_frames without replacement
            selected_frames = sorted(random.sample(frame_files, self.num_frames))
        else:
            # Repeat last frame if not enough
            selected_frames = frame_files + [frame_files[-1]] * (self.num_frames - total_frames)

        frames = []
        for frame_file in selected_frames:
            img_path = os.path.join(video_folder, frame_file)
            img = load_img(img_path, target_size=self.target_size)
            img = img_to_array(img) / 255.0

            if self.augmentations:
                augmented = self.augmentations(image=img)
                img = augmented['image']

            frames.append(img)

        return np.array(frames)


In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import TimeDistributed, LSTM, Dense, GlobalAveragePooling2D, Dropout, BatchNormalization
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.optimizers import Adam

# Base CNN Model
base_model = ResNet50(weights='imagenet', include_top=False, input_shape=(224, 224, 3))



for layer in base_model.layers:
    layer.trainable = False

#Unfreeze more layers for fine-tuning
for layer in base_model.layers[-100:]:
    layer.trainable = True

model = Sequential([
    TimeDistributed(base_model, input_shape=(10, 224, 224, 3)),
    TimeDistributed(GlobalAveragePooling2D()),
    LSTM(256, return_sequences=False, dropout=0.5),
    Dense(256, activation='relu'),
    BatchNormalization(),
    Dropout(0.5),
    Dense(1, activation='sigmoid')
])

model.compile(optimizer=Adam(learning_rate=1e-4),
              loss='binary_crossentropy',
              metrics=['accuracy'])

model.summary()


Model: "sequential_5"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ time_distributed_10             │ (None, 10, 7, 7, 2048) │    23,587,712 │
│ (TimeDistributed)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ time_distributed_11             │ (None, 10, 2048)       │             0 │
│ (TimeDistributed)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_5 (LSTM)                   │ (None, 256)            │     2,360,320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_10 (Dense)                │ (None, 256)            │        65,792 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_5           │ (None, 256)            │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_5 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_11 (Dense)                │ (None, 1)              │           257 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 26,015,105 (99.24 MB)

 Trainable params: 24,578,817 (93.76 MB)

 Non-trainable params: 1,436,288 (5.48 MB)

In [ ]:
# Paths
dataset_path = '/content/train_restructured' # Update to the correct path if different


# Train Generator
train_generator = FrameSequenceGenerator(
    dataset_path=dataset_path,
    batch_size=4,
    num_frames=10,
    shuffle=True,
    augment=True  # apply augmentation during training
)

# Validation Generator (no augmentation)
val_generator = FrameSequenceGenerator(
    dataset_path=dataset_path,
    batch_size=4,
    num_frames=10,
    shuffle=False,
    augment=False
)

# Fit model
model.fit(train_generator, validation_data=val_generator, epochs=20)


Epoch 1/20
164/164 ━━━━━━━━━━━━━━━━━━━━ 281s 945ms/step - accuracy: 0.5280 - loss: 0.9298 - val_accuracy: 0.4174 - val_loss: 0.7877
Epoch 2/20
164/164 ━━━━━━━━━━━━━━━━━━━━ 131s 803ms/step - accuracy: 0.4776 - loss: 0.8241 - val_accuracy: 0.5657 - val_loss: 0.7000
Epoch 3/20
164/164 ━━━━━━━━━━━━━━━━━━━━ 132s 807ms/step - accuracy: 0.5399 - loss: 0.8365 - val_accuracy: 0.5933 - val_loss: 0.7515
Epoch 4/20
164/164 ━━━━━━━━━━━━━━━━━━━━ 132s 807ms/step - accuracy: 0.5251 - loss: 0.8490 - val_accuracy: 0.4862 - val_loss: 0.8030
Epoch 5/20
164/164 ━━━━━━━━━━━━━━━━━━━━ 132s 805ms/step - accuracy: 0.5437 - loss: 0.7898 - val_accuracy: 0.6147 - val_loss: 0.9113
Epoch 6/20
164/164 ━━━━━━━━━━━━━━━━━━━━ 132s 807ms/step - accuracy: 0.4984 - loss: 0.8478 - val_accuracy: 0.6116 - val_loss: 0.8325
Epoch 7/20
164/164 ━━━━━━━━━━━━━━━━━━━━ 132s 805ms/step - accuracy: 0.5343 - loss: 0.7941 - val_accuracy: 0.5398 - val_loss: 0.7535
Epoch 8/20
164/164 ━━━━━━━━━━━━━━━━━━━━ 132s 806ms/step - accuracy: 0.5490 -